In [112]:
# Importing the necessary libararies
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

# Loading the dataset.
customer_transaction_data = pd.read_csv(r"C:\Users\User\Downloads\Fraudulent-Transaction_Detection_For_FinLora_Company\FinLora_Customer_Transaction_Dataset.csv")
customer_transaction_data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\User\\Downloads\\Fraudulent-Transaction_Detection_For_FinLora_Company\\FinLora_Customer_Transaction_Dataset.csv'

In [ ]:
customer_transaction_data.shape

(11400, 26)

In [ ]:
customer_transaction_data.columns


Index(['transaction_id', 'customer_id', 'timestamp', 'home_country',
       'source_currency', 'dest_currency', 'channel', 'amount_src',
       'amount_usd', 'fee', 'exchange_rate_src_to_dest', 'device_id',
       'new_device', 'ip_address', 'ip_country', 'location_mismatch',
       'ip_risk_score', 'kyc_tier', 'account_age_days', 'device_trust_score',
       'chargeback_history_count', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'is_fraud'],
      dtype='object')

In [ ]:

# checking for missing value
customer_transaction_data.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

#Using "isnull" we checking if there are missing values.
And we can confirm that there are:
29 timestamp missing
305 amount_usd 
295 fees
305 ip_address
301 ip_country
300 kyc_tier
295 device_trust_score


In [ ]:
customer_transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  obj

customer_transaction_data["amount_src"].value_counts(dropna=False).head(20)

In [ ]:
customer_transaction_data["amount_src"].value_counts(dropna=False).head(20)

amount_src
100.0     15
102.29     5
115.29     4
71.78      4
102.37     4
105.2      4
119.15     4
250.19     4
261.23     4
94.34      4
54.74      4
167.16     4
58.93      4
121.3      4
82.25      4
128.93     4
144.29     4
99.23      3
295.6      3
160.06     3
Name: count, dtype: int64

In [ ]:
customer_transaction_data["is_fraud"].value_counts(normalize=True).reset_index()

,is_fraud,proportion
0,0,0.912544
1,1,0.087456


In [ ]:
#PROPER DATA CLEANING PROCESSING.
#converting data types of timestamp to datetime
customer_transaction_data["timestamp"] = pd.to_datetime(customer_transaction_data["timestamp"],errors="coerce")
#converting data type of amount_src to float
customer_transaction_data["amount_src"] = pd.to_numeric(customer_transaction_data["amount_src"],errors="coerce")

customer_transaction_data.info()




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             11400 non-null  object             
 1   customer_id                11400 non-null  object             
 2   timestamp                  11339 non-null  datetime64[ns, UTC]
 3   home_country               11400 non-null  object             
 4   source_currency            11400 non-null  object             
 5   dest_currency              11400 non-null  object             
 6   channel                    11400 non-null  object             
 7   amount_src                 11396 non-null  float64            
 8   amount_usd                 11095 non-null  float64            
 9   fee                        11105 non-null  float64            
 10  exchange_rate_src_to_dest  11400 non-null  float64            
 11  de

In [ ]:
# checking for missing value again
customer_transaction_data.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     61
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     4
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [ ]:
# IMPUTATION OF MISSING VARIABLES WITH THE NEEDED VALUES.
#calculating exchange rates per currency
exchange_rates = customer_transaction_data.dropna(subset=["amount_usd","amount_src"]).groupby("source_currency").apply(
    lambda x: (x["amount_usd"] / x["amount_src"]).mean()
).to_dict()
exchange_rates

C:\Users\User\AppData\Local\Temp\ipykernel_26276\3514997686.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  exchange_rates = customer_transaction_data.dropna(subset=["amount_usd","amount_src"]).groupby("source_currency").apply(


{'CAD': 0.7216095926871465,
 'GBP': 1.223441221648679,
 'USD': 0.9838730321259439}

In [ ]:
# Filling the missing value of amount_usd
customer_transaction_data["amount_usd"] = customer_transaction_data.apply(
    lambda row: row["amount_usd"] if pd.notna(row["amount_usd"]) else row["amount_src"] * exchange_rates.get(row["source_currency"],1),
    axis=1
)

In [ ]:
# Filling the missing value of fee variable
if "fee" in customer_transaction_data.columns:
    if "channel" in customer_transaction_data.columns:
        customer_transaction_data["fee"] = customer_transaction_data.groupby("channel")["fee"].transform(lambda y: y.fillna(y.median()))
        customer_transaction_data["fee"] = customer_transaction_data["fee"].fillna(customer_transaction_data["fee"].median())

In [ ]:
# Filling the missing value of ip_country variable
if {"ip_country", "home_country"}.issubset(customer_transaction_data.columns):
     customer_transaction_data["ip_country"] =  customer_transaction_data["ip_country"].fillna(customer_transaction_data["home_country"])

In [ ]:
# Filling the missing value of device_trust_score variable
if "device_trust_score" in customer_transaction_data.columns:
    if {"new_device", "kyc_tier"}.issubset(customer_transaction_data.columns):
        customer_transaction_data["device_trust_score"] = customer_transaction_data.groupby(
            ["new_device", "kyc_tier"]
        )["device_trust_score"].transform(lambda x: x.fillna(x.median()))
    customer_transaction_data['device_trust_score']=customer_transaction_data['device_trust_score'].fillna(customer_transaction_data['device_trust_score'].median())

c:\Users\User\Downloads\Fraudulent-Transaction_Detection_For_FinLora_Company\finLora_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\User\Downloads\Fraudulent-Transaction_Detection_For_FinLora_Company\finLora_env\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [ ]:
customer_transaction_data['kyc_tier'] = customer_transaction_data['kyc_tier'].fillna(customer_transaction_data['kyc_tier'].mode())

In [ ]:
customer_transaction_data["kyc_tier"].value_counts()

kyc_tier
standard       7931
enhanced       1829
low            1039
standrd          72
STANDARD         70
 standard        65
unknown          32
 enhanced        17
ENHANCED         17
enhancd          12
 low              8
LOW               5
 nan              2
NAN               1
Name: count, dtype: int64

In [ ]:
#Dropping the remaining missing value entries
customer_transaction_data.dropna(subset=["timestamp", "amount_src", "ip_address"], inplace=True)

In [ ]:
customer_transaction_data.isnull().sum()

transaction_id               0
customer_id                  0
timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
device_id                    0
new_device                   0
ip_address                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
dtype: int64

In [ ]:
#Carrying out sanity check
#checking for negative values in numeric colomns
negative_counts = {
	"amount_src": (customer_transaction_data["amount_src"] < 0).sum(),
	"amount_usd": (customer_transaction_data["amount_usd"] < 0).sum(),
	"fee": (customer_transaction_data["fee"] < 0).sum(),
	"device_trust_score": (customer_transaction_data["device_trust_score"] < 0).sum(),
	"txn_velocity_1h": (customer_transaction_data["txn_velocity_1h"] < 0).sum(),
	"txn_velocity_24h": (customer_transaction_data["txn_velocity_24h"] < 0).sum(),
	"risk_score_internal": (customer_transaction_data["risk_score_internal"] < 0).sum(),
}
negative_counts

{'amount_src': np.int64(100),
 'amount_usd': np.int64(0),
 'fee': np.int64(90),
 'device_trust_score': np.int64(190),
 'txn_velocity_1h': np.int64(190),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

In [ ]:
customer_transaction_data = customer_transaction_data.drop(customer_transaction_data[
	(customer_transaction_data["amount_src"] < 0) |
	(customer_transaction_data["fee"] < 0) |
	(customer_transaction_data["device_trust_score"] < 0) |
	(customer_transaction_data["txn_velocity_1h"] < 0)].index)



In [ ]:
#checking for negative values in numeric colomns again
negative_counts = {
	"amount_src": (customer_transaction_data["amount_src"] < 0).sum(),
	"amount_usd": (customer_transaction_data["amount_usd"] < 0).sum(),
	"fee": (customer_transaction_data["fee"] < 0).sum(),
	"device_trust_score": (customer_transaction_data["device_trust_score"] < 0).sum(),
	"txn_velocity_1h": (customer_transaction_data["txn_velocity_1h"] < 0).sum(),
	"txn_velocity_24h": (customer_transaction_data["txn_velocity_24h"] < 0).sum(),
	"risk_score_internal": (customer_transaction_data["risk_score_internal"] < 0).sum(),
}
negative_counts

{'amount_src': np.int64(0),
 'amount_usd': np.int64(0),
 'fee': np.int64(0),
 'device_trust_score': np.int64(0),
 'txn_velocity_1h': np.int64(0),
 'txn_velocity_24h': np.int64(0),
 'risk_score_internal': np.int64(0)}

In [ ]:
#Checking for futuristic timestamp
customer_transaction_data[customer_transaction_data["timestamp"] > pd.Timestamp.utcnow()]

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud


In [ ]:
#checking for location mismatch
customer_transaction_data["location_mismatch"].value_counts().reset_index()

,location_mismatch,count
0,False,9047
1,True,1793


In [ ]:
#reviewing the channel variable
customer_transaction_data["channel"].unique()

array(['ATM', 'web', 'mobile', 'WEB', ' web  ', 'MOBILE', 'unknown',
       'mobille', ' mobile  ', 'weeb', 'ATm', ' ATM  '], dtype=object)

In [ ]:
#formatting the channel variable
customer_transaction_data["channel"] = customer_transaction_data["channel"].str.lower().str.strip()

In [ ]:
customer_transaction_data["channel"].unique()

array(['atm', 'web', 'mobile', 'unknown', 'mobille', 'weeb'], dtype=object)

In [ ]:
customer_transaction_data["channel"] = customer_transaction_data["channel"].replace({
	"weeb":"web",
	"mobille":"mobile",
})

In [ ]:
customer_transaction_data["channel"] = customer_transaction_data["channel"].replace({"unknown":np.nan})
customer_transaction_data["channel"].unique()

array(['atm', 'web', 'mobile', nan], dtype=object)

In [ ]:
#reviewing the source cuerrency variable
customer_transaction_data["source_currency"].unique()

array(['USD', 'CAD', 'GBP'], dtype=object)

In [ ]:
#reviewing the destination cuerrency variable
customer_transaction_data["dest_currency"].unique()

array(['CAD', 'MXN', 'CNY', 'EUR', 'INR', 'GBP', 'PHP', 'NGN', 'USD'],
      dtype=object)

In [ ]:
#reviewing the kyc_tier variable
customer_transaction_data["kyc_tier"].unique()

array(['standard', 'enhanced', 'low', ' standard  ', 'standrd',
       ' enhanced  ', 'STANDARD', 'unknown', 'enhancd', ' low  ',
       'ENHANCED', 'LOW'], dtype=object)

In [ ]:
#formatting the kyc_tier variable
customer_transaction_data["kyc_tier"] = customer_transaction_data["kyc_tier"].str.lower().str.strip()
customer_transaction_data["kyc_tier"].unique()

array(['standard', 'enhanced', 'low', 'standrd', 'unknown', 'enhancd'],
      dtype=object)

In [ ]:
#standardizing the wrong varibles
customer_transaction_data["kyc_tier"] = customer_transaction_data["kyc_tier"].replace({
	"standrd":"standard",
	"enhancd":"enhanced",
})
customer_transaction_data["kyc_tier"] = customer_transaction_data["kyc_tier"].replace({"unknown":np.nan})
customer_transaction_data["kyc_tier"].unique()

array(['standard', 'enhanced', 'low', nan], dtype=object)

In [ ]:
customer_transaction_data.isnull().sum()

transaction_id                0
customer_id                   0
timestamp                     0
home_country                  0
source_currency               0
dest_currency                 0
channel                      36
amount_src                    0
amount_usd                    0
fee                           0
exchange_rate_src_to_dest     0
device_id                     0
new_device                    0
ip_address                    0
ip_country                    0
location_mismatch             0
ip_risk_score                 0
kyc_tier                     29
account_age_days              0
device_trust_score            0
chargeback_history_count      0
risk_score_internal           0
txn_velocity_1h               0
txn_velocity_24h              0
corridor_risk                 0
is_fraud                      0
dtype: int64

In [ ]:
customer_transaction_data.dropna(inplace=True)
customer_transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10780 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   transaction_id             10780 non-null  object             
 1   customer_id                10780 non-null  object             
 2   timestamp                  10780 non-null  datetime64[ns, UTC]
 3   home_country               10780 non-null  object             
 4   source_currency            10780 non-null  object             
 5   dest_currency              10780 non-null  object             
 6   channel                    10780 non-null  object             
 7   amount_src                 10780 non-null  float64            
 8   amount_usd                 10780 non-null  float64            
 9   fee                        10780 non-null  float64            
 10  exchange_rate_src_to_dest  10780 non-null  float64            
 11  device_

In [ ]:
#reviewing the home country variable
customer_transaction_data["home_country"].unique()

array(['US', 'CA', 'UK', ' UK  ', ' US  ', 'unknown', ' CA  '],
      dtype=object)

In [ ]:
#formatting the home country variable
customer_transaction_data["home_country"] = customer_transaction_data["home_country"].str.lower().str.strip()
customer_transaction_data["home_country"] = customer_transaction_data["home_country"].replace({"unknown":np.nan})
customer_transaction_data["home_country"].unique()

array(['us', 'ca', 'uk', nan], dtype=object)

In [ ]:
#reviewing the ip_country variable
customer_transaction_data["ip_country"].unique()

array(['US', 'CA', 'UK', ' US  ', 'unknown', ' CA  ', ' UK  '],
      dtype=object)

In [ ]:
#formatting the ip_country variable
customer_transaction_data["ip_country"] = customer_transaction_data["ip_country"].str.lower().str.strip()
customer_transaction_data["ip_country"] = customer_transaction_data["ip_country"].replace({"unknown":np.nan})
customer_transaction_data["ip_country"].unique()

array(['us', 'ca', 'uk', nan], dtype=object)

In [ ]:
customer_transaction_data.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,us,USD,CAD,atm,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,ca,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,us,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,us,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,us,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [ ]:
import os
os.makedirs("../Finlora_Dataset/artifacts")

In [ ]:
customer_transaction_data.to_csv(r"../Finlora_Dataset/artifacts/Cleaned_Data.csv")